# 02 — EDA (Exploratory Data Analysis) TextScope

**PJ:** Yuhanidz (Koordinator Data) · **Minggu 11** · branch `feat/data-quality`

Sesuai panduan tim, notebook ini mengerjakan:
1. Sebaran label per split (train / valid / test)
2. Panjang teks per kelas
3. Kata umum per kelas
4. Visualisasi → disimpan ke `reports/figures/`
5. Bahan untuk **bagian Data** laporan

Tambahan (persiapan Minggu 12–13): kelas yang kosakatanya mirip (kandidat tertukar) dan perbandingan dataset utama vs EmoT.

> **Prasyarat:** `data/processed/` sudah berisi `train.csv`, `valid.csv`, `test.csv`, `emosi_bersih.csv`, `emot_lintas.csv`.
> Analisis **isi teks** (panjang, kata) memakai **train saja** supaya tidak "mengintip" data uji.

## 1. Konfigurasi & muat data

In [ ]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROC    = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
RES_DIR = ROOT / "reports" / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

LABEL = ["anger", "fear", "joy", "love", "sad", "neutral"]
LABEL_EMOT = ["anger", "fear", "joy", "love", "sad"]
WARNA = {"anger": "#E45756", "fear": "#9D6CD0", "joy": "#F2B134",
         "love": "#F28EBC", "sad": "#4C78A8", "neutral": "#8C8C8C"}

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

train = pd.read_csv(PROC / "train.csv")
valid = pd.read_csv(PROC / "valid.csv")
test  = pd.read_csv(PROC / "test.csv")
bersih = pd.read_csv(PROC / "emosi_bersih.csv")
emot   = pd.read_csv(PROC / "emot_lintas.csv")

for nama, d in [("train", train), ("valid", valid), ("test", test), ("emot", emot)]:
    d["split"] = nama
    d["text"] = d["text"].astype(str)
    d["n_kata"] = d["text"].str.split().str.len()
    d["n_char"] = d["text"].str.len()

semua = pd.concat([train, valid, test], ignore_index=True)
print(f"train={len(train):,}  valid={len(valid):,}  test={len(test):,}  | emosi_bersih={len(bersih):,}  | emot_lintas={len(emot):,}")
semua.head()

## 2. Pemeriksaan cepat kualitas (sanity check)

In [ ]:
def kunci(s):
    return s.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

print("Total baris split   :", f"{len(semua):,}", "| sama dengan emosi_bersih:", len(semua) == len(bersih))
print("Nilai kosong        :", int(semua[["text", "label"]].isna().sum().sum()))
print("Label di luar skema :", sorted(set(semua["label"]) - set(LABEL)))

k_tr, k_va, k_te = (set(kunci(d["text"])) for d in (train, valid, test))
print("Overlap train∩valid :", len(k_tr & k_va))
print("Overlap train∩test  :", len(k_tr & k_te))
print("Overlap valid∩test  :", len(k_va & k_te))
print("Duplikat dalam split:", int(kunci(semua["text"]).duplicated().sum()))

## 3. Sebaran label per split

In [ ]:
tabel_n = (semua.pivot_table(index="label", columns="split", values="text", aggfunc="count")
           [["train", "valid", "test"]].reindex(LABEL))
tabel_n["total"] = tabel_n.sum(axis=1)
tabel_p = (tabel_n[["train", "valid", "test", "total"]] / tabel_n[["train", "valid", "test", "total"]].sum() * 100).round(2)

print("Jumlah baris:"); display(tabel_n)
print("Proporsi (%) — harus nyaris sama antar split bila stratified:"); display(tabel_p)

tabel_n.to_csv(RES_DIR / "eda_sebaran_label_jumlah.csv")
tabel_p.to_csv(RES_DIR / "eda_sebaran_label_persen.csv")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

tabel_n["total"].plot.bar(ax=ax[0], color=[WARNA[l] for l in LABEL], width=0.75)
for i, v in enumerate(tabel_n["total"]):
    ax[0].text(i, v + tabel_n["total"].max() * 0.01, f"{int(v):,}", ha="center", fontsize=9)
ax[0].set_title("Jumlah teks per kelas (semua split)"); ax[0].set_xlabel(""); ax[0].tick_params(axis="x", rotation=0)

x = np.arange(len(LABEL)); w = 0.27
for j, sp in enumerate(["train", "valid", "test"]):
    ax[1].bar(x + (j - 1) * w, tabel_p[sp].values, w, label=sp)
ax[1].set_xticks(x); ax[1].set_xticklabels(LABEL)
ax[1].set_title("Proporsi kelas per split (%)"); ax[1].set_ylabel("%"); ax[1].legend()

plt.tight_layout(); plt.savefig(FIG_DIR / "eda_distribusi_label.png", dpi=150); plt.show()

# Ketimpangan kelas + pembanding "tebak kelas mayoritas"
jml = tabel_n["total"]
mayoritas = jml.idxmax(); p = jml.max() / jml.sum()
f1_mayoritas = (2 * p * 1 / (p + 1)) / len(LABEL)        # F1 kelas mayoritas (recall=1, precision=p), kelas lain 0
print(f"Kelas terbesar : {mayoritas} ({jml.max():,})  |  terkecil: {jml.idxmin()} ({jml.min():,})")
print(f"Rasio ketimpangan terbesar:terkecil = {jml.max() / jml.min():.2f} : 1")
print(f"Pembanding: model yang selalu menebak '{mayoritas}' -> akurasi {p:.3f}, Macro F1 ≈ {f1_mayoritas:.3f}")

**Catatan temuan (isi setelah melihat hasil):**
- Apakah proporsi antar split sama (stratified berhasil)?
- Kelas mana yang minoritas? Perlu class weighting di Minggu 12 (Linda)?
- Karena kelas tidak seimbang, **Macro F1** memang metrik yang tepat (akurasi saja menyesatkan).

## 4. Panjang teks per kelas (train)

In [ ]:
desk = (train.groupby("label")["n_kata"].describe(percentiles=[.5, .9, .95]).reindex(LABEL).round(1))
print("Jumlah kata per teks:"); display(desk)
desk.to_csv(RES_DIR / "eda_panjang_teks_per_kelas.csv")

pers = train["n_kata"].quantile([.5, .9, .95, .99, 1.0])
print("\nPersentil jumlah kata (semua kelas, train):")
print(pers.to_string())
print("\nTeks <= 3 kata  :", int((train["n_kata"] <= 3).sum()), f"({(train['n_kata'] <= 3).mean() * 100:.1f}%)")
print("Teks >= 100 kata:", int((train["n_kata"] >= 100).sum()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

ax[0].hist(train["n_kata"].clip(upper=100), bins=40, color="#4C78A8")
ax[0].set_title("Distribusi panjang teks (kata, dipotong di 100)")
ax[0].set_xlabel("jumlah kata"); ax[0].set_ylabel("jumlah teks")

data_box = [train.loc[train["label"] == l, "n_kata"].clip(upper=100) for l in LABEL]
bp = ax[1].boxplot(data_box, patch_artist=True, showfliers=False)
ax[1].set_xticks(range(1, len(LABEL) + 1)); ax[1].set_xticklabels(LABEL)
for patch, l in zip(bp["boxes"], LABEL):
    patch.set_facecolor(WARNA[l]); patch.set_alpha(0.8)
ax[1].set_title("Panjang teks per kelas (kata)"); ax[1].set_ylabel("jumlah kata")

plt.tight_layout(); plt.savefig(FIG_DIR / "eda_panjang_teks.png", dpi=150); plt.show()

**Catatan temuan:** apakah ada kelas yang teksnya jauh lebih pendek/panjang (mis. `neutral`)? Panjang teks yang berbeda antar kelas bisa jadi "jalan pintas" yang dipelajari model.
Persentil di atas berguna untuk Chila memilih `max_length` tokenizer transformer (token subword biasanya lebih banyak dari jumlah kata).

## 5. Kata umum per kelas (train)

In [ ]:
STOPWORD = set("""
yang dan di ke dari ini itu untuk dengan pada adalah juga atau tidak ga gak nggak tak ya aja saja sih deh dong lah kok kan
nya ada akan sudah udah belum bisa dapat jadi karena sama lebih sangat banget yg dg dgn utk tp tapi dalam oleh saya aku kamu
kita kami mereka dia anda gue gw lu lo kau mu ku buat bgt apa siapa bagaimana kenapa mengapa kalau kalo jika agar supaya
sebagai para pun hanya masih telah sedang harus mau lagi lg nih tu tuh se per kalian ini itu sebuah semua setiap
""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

train["tok"] = train["text"].map(tokenisasi)
train["tok_bersih"] = train["tok"].map(lambda t: [w for w in t if len(w) > 1 and w not in STOPWORD])

hitung = {l: Counter(w for t in train.loc[train["label"] == l, "tok_bersih"] for w in t) for l in LABEL}

TOP = 12
baris = []
for l in LABEL:
    for w, c in hitung[l].most_common(TOP):
        baris.append({"label": l, "kata": w, "jumlah": c})
top_kata = pd.DataFrame(baris)
top_kata.to_csv(RES_DIR / "eda_kata_umum_per_kelas.csv", index=False)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, l in zip(axes.ravel(), LABEL):
    d = top_kata[top_kata["label"] == l].iloc[::-1]
    ax.barh(d["kata"], d["jumlah"], color=WARNA[l])
    ax.set_title(l)
plt.suptitle("Kata paling umum per kelas (tanpa stopword)", y=1.0)
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_kata_umum.png", dpi=150); plt.show()

### Kata paling *khas* per kelas
Kata umum sering sama di semua kelas. Di sini dipakai rasio log-peluang (kelas ini vs kelas lain) supaya yang muncul adalah kata yang **membedakan** kelas.

In [ ]:
total = Counter()
for c in hitung.values():
    total.update(c)

def kata_khas(kelas, min_count=5, top=10):
    n_k = sum(hitung[kelas].values()); n_r = sum(total.values()) - n_k
    skor = {}
    for w, c in hitung[kelas].items():
        if c < min_count:
            continue
        c_r = total[w] - c
        skor[w] = np.log((c + 1) / (n_k + 1)) - np.log((c_r + 1) / (n_r + 1))
    return [w for w, _ in sorted(skor.items(), key=lambda x: -x[1])[:top]]

khas = pd.DataFrame({l: pd.Series(kata_khas(l)) for l in LABEL})
khas.to_csv(RES_DIR / "eda_kata_khas_per_kelas.csv", index=False)
display(khas)

**Catatan temuan:** apakah kata khasnya masuk akal secara emosi? Kalau muncul kata aneh (nama orang, topik tertentu, sisa tanda baca), itu tanda **bias topik** pada dataset — penting dilaporkan.

## 6. Ciri permukaan teks per kelas (train)
Emoji, tanda seru/tanya, angka, dan HURUF KAPITAL — sering berbeda antar emosi, dan relevan untuk keputusan preprocessing.

In [ ]:
RE_EMOJI = re.compile(r"[\U0001F300-\U0001FAFF☀-➿]")

fitur = pd.DataFrame({
    "label": train["label"],
    "ada '!'": train["text"].str.contains("!", regex=False),
    "ada '?'": train["text"].str.contains("?", regex=False),
    "ada angka": train["text"].str.contains(r"\d", regex=True),
    "ada emoji": train["text"].map(lambda s: bool(RE_EMOJI.search(s))),
    "HURUF KAPITAL SEMUA": train["text"].map(lambda s: s.isupper()),
    "<= 3 kata": train["n_kata"] <= 3,
})
ciri = (fitur.groupby("label").mean().reindex(LABEL) * 100).round(1)
print("Persentase teks yang memiliki ciri (%):")
display(ciri)
ciri.to_csv(RES_DIR / "eda_ciri_permukaan.csv")

## 7. Kelas yang kosakatanya mirip (kandidat tertukar)
Persiapan analisis teks sulit Minggu 12 dan analisis error Minggu 13. Diukur dengan **Jaccard** pada 100 kata teratas tiap kelas — makin tinggi, makin mirip.

In [ ]:
top100 = {l: set(w for w, _ in hitung[l].most_common(100)) for l in LABEL}
jac = pd.DataFrame(index=LABEL, columns=LABEL, dtype=float)
for a in LABEL:
    for b in LABEL:
        jac.loc[a, b] = len(top100[a] & top100[b]) / len(top100[a] | top100[b])

fig, ax = plt.subplots(figsize=(5.8, 5))
im = ax.imshow(jac.values.astype(float), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(LABEL))); ax.set_xticklabels(LABEL, rotation=45, ha="right")
ax.set_yticks(range(len(LABEL))); ax.set_yticklabels(LABEL)
for i in range(len(LABEL)):
    for j in range(len(LABEL)):
        ax.text(j, i, f"{jac.values[i, j]:.2f}", ha="center", va="center",
                color="white" if jac.values[i, j] > 0.55 else "black", fontsize=8)
ax.set_title("Kemiripan kosakata (Jaccard top-100 kata)")
plt.colorbar(im, fraction=0.046); plt.tight_layout()
plt.savefig(FIG_DIR / "eda_kemiripan_kelas.png", dpi=150); plt.show()

pasangan = (jac.where(np.triu(np.ones(jac.shape), k=1).astype(bool)).stack()
            .sort_values(ascending=False).head(5).round(3))
print("5 pasangan kelas paling mirip:"); print(pasangan.to_string())
jac.round(3).to_csv(RES_DIR / "eda_kemiripan_kelas.csv")

**Catatan temuan:** bandingkan dengan dugaan di panduan (`neutral↔sad`, `love↔joy`). Pasangan yang mirip di sini kemungkinan besar juga paling sering tertukar di confusion matrix Minggu 13.

## 8. Teks sangat pendek (kandidat ambigu)

In [ ]:
pendek = train[train["n_kata"] <= 3]
print("Jumlah teks <= 3 kata per kelas:")
print(pendek["label"].value_counts().reindex(LABEL).fillna(0).astype(int).to_string())
print("\nContoh (3 per kelas) — samarkan/anonimkan lagi sebelum dikutip di laporan:")
for l in LABEL:
    contoh = pendek[pendek["label"] == l]["text"].head(3).tolist()
    print(f"  [{l}] {contoh}")

## 9. Dataset utama vs EmoT (persiapan evaluasi lintas domain)

In [ ]:
# Sebaran label: utama (tanpa neutral, dinormalisasi) vs EmoT
utama5 = train[train["label"].isin(LABEL_EMOT)]["label"].value_counts(normalize=True).reindex(LABEL_EMOT) * 100
emot5  = emot["label"].value_counts(normalize=True).reindex(LABEL_EMOT) * 100
bandingan = pd.DataFrame({"utama (train, tanpa neutral) %": utama5.round(1), "EmoT %": emot5.round(1)})
display(bandingan)
bandingan.to_csv(RES_DIR / "eda_utama_vs_emot_label.csv")

# Kosakata: seberapa banyak kata EmoT yang tidak pernah terlihat di train?
vocab_train = set(w for t in train["tok"] for w in t)
emot["tok"] = emot["text"].map(tokenisasi)
valid["tok"] = valid["text"].map(tokenisasi)

def oov(df):
    token = [w for t in df["tok"] for w in t]
    tipe = set(token)
    return (np.mean([w not in vocab_train for w in token]) * 100,
            np.mean([w not in vocab_train for w in tipe]) * 100)

oov_valid = oov(valid); oov_emot = oov(emot)
print(f"\nOOV terhadap kosakata train — token / tipe kata:")
print(f"  valid (domain sama)   : {oov_valid[0]:.1f}% / {oov_valid[1]:.1f}%")
print(f"  EmoT  (lintas domain) : {oov_emot[0]:.1f}% / {oov_emot[1]:.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(LABEL_EMOT)); w = 0.38
ax[0].bar(x - w / 2, utama5.values, w, label="utama (train)", color="#4C78A8")
ax[0].bar(x + w / 2, emot5.values, w, label="EmoT", color="#F58518")
ax[0].set_xticks(x); ax[0].set_xticklabels(LABEL_EMOT); ax[0].set_ylabel("%"); ax[0].legend()
ax[0].set_title("Sebaran label (5 kelas bersama)")

ax[1].hist(train["n_kata"].clip(upper=80), bins=40, alpha=0.6, density=True, label="utama (train)", color="#4C78A8")
ax[1].hist(emot["n_kata"].clip(upper=80), bins=40, alpha=0.6, density=True, label="EmoT", color="#F58518")
ax[1].set_title("Panjang teks (kata, dipotong di 80)"); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_utama_vs_emot.png", dpi=150); plt.show()

print(f"\nMedian kata: utama={train['n_kata'].median():.0f}  EmoT={emot['n_kata'].median():.0f}")

**Catatan temuan:** OOV EmoT yang jauh lebih tinggi daripada valid = bukti kuantitatif beda domain (gaya bahasa/kosakata). Ini bahan untuk penjelasan hasil lintas domain di bagian Eksperimen (Chila).

## 10. Ringkasan untuk bagian **Data** laporan

In [ ]:
n_tot = len(semua)
print("=== RINGKASAN DATA (salin ke laporan) ===")
print(f"Dataset utama (bersih)   : {n_tot:,} teks, 6 kelas")
print(f"Split (stratified)       : train {len(train):,} / valid {len(valid):,} / test {len(test):,}")
print(f"Kelas terbesar / terkecil: {jml.idxmax()} ({jml.max():,}) / {jml.idxmin()} ({jml.min():,})  -> rasio {jml.max() / jml.min():.2f}:1")
print(f"Pembanding mayoritas     : akurasi {p:.3f}, Macro F1 ≈ {f1_mayoritas:.3f}")
print(f"Panjang teks (kata)      : median {train['n_kata'].median():.0f}, p95 {train['n_kata'].quantile(.95):.0f}, maks {train['n_kata'].max()}")
print(f"Teks <= 3 kata           : {(train['n_kata'] <= 3).mean() * 100:.1f}% dari train")
print(f"EmoT (lintas domain)     : {len(emot):,} teks, 5 kelas (tanpa neutral); OOV token {oov_emot[0]:.1f}% vs valid {oov_valid[0]:.1f}%")
print(f"Pasangan kelas paling mirip: {pasangan.index[0][0]}↔{pasangan.index[0][1]} ({pasangan.iloc[0]:.2f})")
print("\nGrafik tersimpan di reports/figures/ (eda_*.png); tabel di reports/results/ (eda_*.csv)")

### Checklist Minggu 11 (Yuhanidz)
- [ ] EDA selesai (sebaran label per split, panjang teks per kelas, kata umum per kelas)
- [ ] Grafik tersimpan di `reports/figures/`
- [ ] Bagian **Data** laporan ditulis: sumber, jumlah, sebaran, preprocessing, lisensi
- [ ] Commit ke branch `feat/data-quality` → pull request → review cadangan (Zura) → merge oleh Linda

> Sumber & lisensi: Emotion Dataset from Indonesian Public Opinion (CC-BY-NC-ND 4.0); EmoT dari IndoNLU (cek lisensi di repo IndoNLU).
> Sebelum mengutip contoh teks di laporan, pastikan sudah dianonimkan (aturan kerja no. 6).